Tek head'i multi-head attention'a çevir: aynı boyutu n_head tane küçük head'e böl, çıktıları birleştir. Val loss'u geçen haftaki tek head'le yan yana koy. (1:21:59 - 1:24:25)

In [5]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import pandas as pd

In [ ]:
batch_size = 32
block_size = 8
max_iters = 3000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32
n_head=4
torch.manual_seed(42)

In [7]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [8]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [9]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [10]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [11]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


In [12]:
print(f"Cihaz: {device}, Vocab Size: {vocab_size}, n_embd: {n_embd}")

Cihaz: cuda, Vocab Size: 65, n_embd: 32


### Self-Attention Head Sınıfı

In [20]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # Attention score
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5) # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)

        # ağırlıklı toplam
        v = self.value(x) # (B, T, head_size)
        out = wei @ v     # (B, T, head_size)
        return out

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) #(B, T, C)
        out = self.proj(out)
        return out

### Self-attention'lı Bigram Modeli

In [ ]:
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = MultiHeadAttention(n_head,n_embd // n_head)         
        self.lm_head = nn.Linear(n_embd, vocab_size)


    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, n_embd)
        x = tok_emb + pos_emb # (B, T, n_embd)

        # Self-attention uygula
        x = self.sa_head(x) # (B, T, n_embd)
        x = self.ln_f(x) # (B,T,C)
        # Logitleri üret
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx: (B, T) boyutunda mevcut context
        for _ in range(max_new_tokens):
            # position embedding sınırını aşmamak için son 8 karaktere bakıyoruz
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

### 4. Modelin Eğitilmesi

In [32]:
torch.manual_seed(42)
model = BigramLanguageModel().to(device)
print(f"Toplam parameter: {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"{iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

Toplam parameter: 8,673
0: train loss 4.4859, val loss 4.4817
300: train loss 2.6567, val loss 2.6756
600: train loss 2.5012, val loss 2.5025
900: train loss 2.4308, val loss 2.4286
1200: train loss 2.3819, val loss 2.3850
1500: train loss 2.3446, val loss 2.3779
1800: train loss 2.3254, val loss 2.3466
2100: train loss 2.3093, val loss 2.3221
2400: train loss 2.2864, val loss 2.3192
2700: train loss 2.2826, val loss 2.2937
2999: train loss 2.2621, val loss 2.2995


### Örnek metin üretimi

In [33]:
final_losses = estimate_loss(model)
train_loss4 = final_losses['train']
val_loss4 = final_losses['val']

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_text = decode(model.generate(context, max_new_tokens=500)[0].tolist())

print(generated_text)


I
Ou Ceatnus thim the thearmd tharnd he bureckn,
Os I for of will lapiwist.

TIThak than,
TAnt uprt, of nis adevy ditle cont Et im thy
Sill anmean theur that ghe me machan fl Coust dus to-istwurore canse wince my, worss thee?

matorest;
RICORD ming I he wlars oln not in Whas this ther'd ban noow
Far tho be's lore wy byd pe, Rpel me the gosts I mearung.

COr goy sock:
ToNow an te sheir I wow the p'whand nowe bivou Yul:
K:
Thar!
Thou hou my for to aasssostal?

Rhis the ake 's and barsst ut wnoide.
